# Compression-02 — Huffman canonique : des longueurs aux tables de décodage

**Navigation** : [Index de la série](README.md) · [Version Python](Compression-02-HuffmanCanonique-Python.ipynb) · [Compression-01 — le socle préfixe](Compression-01-ShannonFano-Prefixe-Python.ipynb)

> **Public : Découverte.** Prérequis : Compression-01 (codes préfixes, inégalité de Kraft, arbre de Huffman). Ce carnet reconstruit la forme canonique depuis les longueurs, bâtit une table de décodage $2^L$ et vérifie un aller-retour exact sur un vrai extrait du *Horla*.
>
> **Durée estimée** : 40 minutes. **Environnement** : .NET 9, BCL uniquement.
>
> **Objectifs** : mesurer Huffman sur des octets réels ; reconstruire des codes canoniques ; comprendre le compromis table/décodage ; comparer les coûts complets et mesurer les débits.
>
> **Parcours** : corpus de 128 Kio · arbre et longueurs · canonisation · table $2^L$ · aller-retour octet pour octet · débits et limites. Trois exercices sont répartis dans le parcours, au fil des notions.


In [1]:
using System;
using System.Collections.Generic;
using System.Diagnostics;
using System.IO;
using System.Linq;
using System.Numerics;
using System.Security.Cryptography;
using System.Text;

Console.WriteLine($"Runtime du noyau : .NET {Environment.Version} ; carnet compatible .NET 9, BCL uniquement.");

The below script needs to be able to find the current output cell; this is an easy method to get it.

Runtime du noyau : .NET 10.0.14 ; carnet compatible .NET 9, BCL uniquement.


## 1. Le corpus : le même extrait binaire, 128 Kio

Nous lisons les 131 072 premiers octets du fichier du *Horla* déjà présent dans le dépôt. Le SHA-256 de l’extrait fixe le corpus sans imprimer de chemin local. Le traitement porte sur les octets UTF-8 tels quels, pas sur des caractères Unicode.

In [2]:
var horlaPath = Path.GetFullPath("../GenAI/Audio/04-Applications/datasets/le_horla_1887.txt");
var corpusComplet = File.ReadAllBytes(horlaPath);
var extrait = corpusComplet.Take(131072).ToArray();
var hashOriginal = Convert.ToHexString(SHA256.HashData(extrait)).ToLowerInvariant();
var frequences = new long[256];
foreach (byte b in extrait) frequences[b]++;
var alphabet = Enumerable.Range(0, 256).Where(i => frequences[i] > 0).ToArray();
Console.WriteLine($"Corpus : {Path.GetFileName(horlaPath)} | extrait : {extrait.Length} octets | SHA-256 : {hashOriginal}");
Console.WriteLine($"Alphabet observé : {alphabet.Length} octets distincts sur 256");
Console.WriteLine($"Octet espace : {frequences[0x20]} occurrences ({100.0 * frequences[0x20] / extrait.Length:F1} %)");

Corpus : le_horla_1887.txt | extrait : 131072 octets | SHA-256 : 248fb4ec30c05351198a40c81969624d447fb8ad36e89d6bb5f8ce6084090704


Alphabet observé : 88 octets distincts sur 256


Octet espace : 20051 occurrences (15.3 %)


### Lecture

L’histogramme porte sur chaque octet du fichier UTF-8, y compris les octets de continuation des caractères accentués. Son empreinte permet de confronter les deux implémentations sur les mêmes données.

## 2. Construire l’arbre et mesurer ses longueurs

Huffman fusionne à chaque étape les deux nœuds les plus légers. Un compteur de séquence rend le départage déterministe dans cette exécution ; d’autres politiques de départage peuvent produire un arbre différent pour les mêmes fréquences ex æquo. La longueur pondérée optimale reste la mesure à comparer.

In [3]:
sealed class HuffNode
{
    public int Symbol;
    public HuffNode Left = null!;
    public HuffNode Right = null!;
    public long Weight;
    public bool IsLeaf => Left is null && Right is null;
}

static (Dictionary<int, int> Lengths, Dictionary<int, string> Codes, HuffNode Root)
    BuildHuffman(long[] counts)
{
    var queue = new PriorityQueue<HuffNode, (long Weight, int Order)>();
    int order = 0;
    for (int symbol = 0; symbol < counts.Length; symbol++)
    {
        if (counts[symbol] > 0)
        {
            var leaf = new HuffNode { Symbol = symbol, Weight = counts[symbol] };
            queue.Enqueue(leaf, (counts[symbol], order++));
        }
    }

    if (queue.Count == 0)
        throw new ArgumentException("Alphabet vide");
    if (queue.Count == 1)
    {
        var only = queue.Dequeue();
        return (new() { { only.Symbol, 1 } }, new() { { only.Symbol, "0" } }, only);
    }

    while (queue.Count > 1)
    {
        var left = queue.Dequeue();
        var right = queue.Dequeue();
        var parent = new HuffNode { Left = left, Right = right, Weight = left.Weight + right.Weight };
        queue.Enqueue(parent, (parent.Weight, order++));
    }

    var root = queue.Dequeue();
    var lengths = new Dictionary<int, int>();
    var codes = new Dictionary<int, string>();
    void Visit(HuffNode node, string path)
    {
        if (node.IsLeaf)
        {
            lengths[node.Symbol] = path.Length;
            codes[node.Symbol] = path;
            return;
        }
        Visit(node.Left, path + "0");
        Visit(node.Right, path + "1");
    }
    Visit(root, "");
    return (lengths, codes, root);
}

var emptyAlphabetRejected = false;
try
{
    BuildHuffman(new long[256]);
}
catch (ArgumentException error) when (error.Message.Contains("Alphabet vide"))
{
    emptyAlphabetRejected = true;
}
if (!emptyAlphabetRejected)
    throw new Exception("Un alphabet vide doit être rejeté explicitement.");
Console.WriteLine("Alphabet vide : rejet explicite ; aucun code n’est défini pour un corpus vide.");

var arbreResultat = BuildHuffman(frequences);
var longueursArbre = arbreResultat.Lengths;
var codesArbre = arbreResultat.Codes;
var racine = arbreResultat.Root;
var totalBitsArbre = longueursArbre.Sum(kv => frequences[kv.Key] * kv.Value);
var entropie = -alphabet.Sum(symbol =>
{
    double probability = (double)frequences[symbol] / extrait.Length;
    return probability * Math.Log2(probability);
});
var moyenne = (double)totalBitsArbre / extrait.Length;
var lMax = longueursArbre.Values.Max();
Console.WriteLine($"L max : {lMax} | moyenne : {moyenne:F3} bits/octet | entropie : {entropie:F3} | écart : {moyenne - entropie:F3}");
Console.WriteLine($"Flux théorique : {totalBitsArbre} bits, soit {(totalBitsArbre + 7) / 8} octets avant en-tête.");

Alphabet vide : rejet explicite ; aucun code n’est défini pour un corpus vide.


L max : 15 | moyenne : 4.655 bits/octet | entropie : 4.620 | écart : 0.035


Flux théorique : 610195 bits, soit 76275 octets avant en-tête.


### Lecture

La longueur moyenne du code entier se situe au-dessus de l’entropie, qui est la borne pour un code idéal. La longueur maximale $L$ fixe ensuite la taille de la table. Les octets ayant la même fréquence peuvent permuter dans l’arbre selon le départage, sans changer le coût total optimal.

### Exemple 1 — Visualiser l’arbre par ses longueurs

Le graphique SVG statique montre combien de symboles ont chaque longueur. Il est généré localement, sans JavaScript ni bibliothèque externe.

In [4]:
var repartitionLongueurs = longueursArbre.Values
    .GroupBy(length => length)
    .OrderBy(group => group.Key)
    .ToArray();
int svgWidth = 720;
int svgHeight = 300;
int plotBottom = 245;
int plotTop = 35;
int maxCount = repartitionLongueurs.Max(group => group.Count());
var barSvg = new StringBuilder(
    $"<svg xmlns='http://www.w3.org/2000/svg' width='{svgWidth}' height='{svgHeight}' viewBox='0 0 {svgWidth} {svgHeight}' role='img' aria-label='Distribution des longueurs de code'>");
barSvg.Append("<rect width='100%' height='100%' fill='white'/><text x='360' y='22' text-anchor='middle' font-family='sans-serif' font-size='16'>Longueurs de Huffman — extrait du Horla</text>");
foreach (var group in repartitionLongueurs)
{
    double x = 65 + group.Key * 38;
    double barHeight = (plotBottom - plotTop) * (double)group.Count() / maxCount;
    barSvg.Append($"<rect x='{x:F0}' y='{plotBottom - barHeight:F1}' width='32' height='{barHeight:F1}' fill='#4C72B0'/>");
    barSvg.Append($"<text x='{x + 16:F0}' y='{plotBottom + 18}' text-anchor='middle' font-family='sans-serif' font-size='11'>{group.Key}</text>");
    barSvg.Append($"<text x='{x + 16:F0}' y='{plotBottom - barHeight - 5:F1}' text-anchor='middle' font-family='sans-serif' font-size='10'>{group.Count()}</text>");
}
barSvg.Append("<text x='360' y='285' text-anchor='middle' font-family='sans-serif' font-size='12'>Longueur du code (bits)</text><text transform='translate(18 150) rotate(-90)' text-anchor='middle' font-family='sans-serif' font-size='12'>Nombre d’octets distincts</text></svg>");
public record HuffmanSvg(string Content);
Microsoft.DotNet.Interactive.Formatting.Formatter.Register<HuffmanSvg>(
    (value, writer) => writer.Write(value.Content), "text/html");
display(new HuffmanSvg(barSvg.ToString()));
Console.WriteLine($"Distribution calculée pour {repartitionLongueurs.Length} longueurs distinctes.");

Longueurs de Huffman — extrait du Horla 3 2 4 6 5 5 6 7 7 8 8 2 9 10 10 12 11 11 12 6 13 3 14 4 15 12 Longueur du code (bits) Nombre d’octets distincts

Distribution calculée pour 13 longueurs distinctes.


### Lecture

Les longueurs courtes concentrent généralement davantage de symboles fréquents ; une queue représente les octets rares. La valeur de $L$ mesurée ici dimensionne la table directe.

## 3. La forme canonique (Schwartz et Kallick, 1964)

Les longueurs déterminent un code canonique indépendamment de l’arbre : trier par longueur puis par valeur d’octet, attribuer le premier mot, puis incrémenter et décaler à gauche lorsque la longueur augmente. Le flux utilise des bits empaquetés en octets : il ne matérialise pas une chaîne de caractères `'0'`/`'1'`.

In [5]:
static Dictionary<int, BigInteger> CanonicalCodes(Dictionary<int, int> lengths)
{
    var ordered = lengths.OrderBy(item => item.Value).ThenBy(item => item.Key).ToArray();
    var result = new Dictionary<int, BigInteger>();
    BigInteger code = 0;
    int previousLength = 0;
    foreach (var item in ordered)
    {
        int length = item.Value;
        if (length <= 0)
            throw new ArgumentException("Longueur non positive");
        code <<= length - previousLength;
        result[item.Key] = code;
        code++;
        previousLength = length;
    }
    return result;
}

var codesCanoniques = CanonicalCodes(longueursArbre);
var bitsCanoniques = longueursArbre.Sum(kv => frequences[kv.Key] * (long)kv.Value);
var kraft = longueursArbre.Values.Sum(length => Math.Pow(2, -length));
Console.WriteLine($"Coût arbre = {totalBitsArbre} bits ; coût canonique = {bitsCanoniques} bits ; Kraft = {kraft:F12}");
Console.WriteLine("Octet | fréquence | longueur | mot arbre | mot canonique");
foreach (int symbol in alphabet.OrderByDescending(symbol => frequences[symbol]).Take(8))
{
    string shown = symbol >= 32 && symbol < 127 ? ((char)symbol).ToString() : $"\\x{symbol:x2}";
    string canonical = Convert.ToString((long)codesCanoniques[symbol], 2)
        .PadLeft(longueursArbre[symbol], '0');
    Console.WriteLine($"{shown,6} | {frequences[symbol],9} | {longueursArbre[symbol],8} | {codesArbre[symbol],9} | {canonical}");
}
if (bitsCanoniques != totalBitsArbre || Math.Abs(kraft - 1) > 1e-12)
    throw new Exception("Invariants canoniques violés");

Coût arbre = 610195 bits ; coût canonique = 610195 bits ; Kraft = 1.000000000000


Octet | fréquence | longueur | mot arbre | mot canonique


       |     20051 |        3 |       110 | 000


     e |     14446 |        3 |       011 | 001


     s |      7773 |        4 |      1001 | 0111


     a |      7284 |        4 |      1000 | 0100


     i |      6928 |        4 |      0100 | 0101


     t |      6477 |        4 |      0011 | 1000


     u |      6384 |        4 |      0001 | 1001


     n |      6281 |        4 |      0000 | 0110


### Lecture

Les mots peuvent différer de ceux de l’arbre, mais leurs longueurs restent identiques. L’en-tête peut donc contenir les longueurs par symbole ; le décodeur reconstruit les mêmes mots avec la procédure canonique. Kraft égal à un indique ici un code complet.

### Exercice 1 — Ces longueurs décrivent-elles un code ?

Écrivez `CanBePrefix` selon l’inégalité de Kraft–McMillan. Testez une famille complète, une famille incomplète réalisable et une famille impossible.

In [6]:
static bool CanBePrefix(int[] lengths)
{
    // TODO étudiant : calculer la somme de Kraft, puis vérifier qu’elle ne dépasse pas 1.
    // Indice : comparer en double ou multiplier par 2^L pour éviter les fractions.
    return false;
}

var kraftVerdicts = new[]
{
    CanBePrefix(new[] { 1, 2, 3, 3 }),
    CanBePrefix(new[] { 2, 2, 2 }),
    CanBePrefix(new[] { 1, 2, 2, 2 })
};
if (kraftVerdicts.SequenceEqual(new[] { true, true, false }))
    Console.WriteLine($"Exercice 1 : OK — {string.Join(", ", kraftVerdicts)}");
else
    Console.WriteLine("Exercice à compléter");

Exercice à compléter


## 4. Décoder sans arbre : une table à $2^L$ entrées

Un index formé des $L$ prochains bits sélectionne symbole et longueur réelle. Un code de longueur $\ell$ remplit $2^{L-\ell}$ entrées. La table joue le compromis temps/mémoire : deux tableaux d’octets utilisent $2·2^L$ octets, chaque incrément de $L$ double leur coût.

### Exemple 2 — Une petite table canonique

La distribution jouet produit les longueurs 1, 2, 3, 3 et permet d’inspecter chaque entrée.

In [7]:
static (byte[] Symbols, byte[] Lengths) BuildDecodeTable(
    Dictionary<int, BigInteger> codes, Dictionary<int, int> lengths, int maxLength)
{
    if (maxLength > 24)
        throw new ArgumentOutOfRangeException(nameof(maxLength), "Table pédagogique plafonnée à 2^24 entrées");

    int size = 1 << maxLength;
    var symbols = new byte[size];
    var tableLengths = new byte[size];
    var filled = new bool[size];
    if (codes.Count == 1)
    {
        var only = codes.Single();
        Array.Fill(symbols, (byte)only.Key);
        Array.Fill(tableLengths, (byte)lengths[only.Key]);
        return (symbols, tableLengths);
    }

    foreach (var item in codes)
    {
        int length = lengths[item.Key];
        int baseIndex = (int)item.Value << (maxLength - length);
        int span = 1 << (maxLength - length);
        for (int index = baseIndex; index < baseIndex + span; index++)
        {
            if (filled[index])
                throw new Exception("Collision de préfixes");
            symbols[index] = (byte)item.Key;
            tableLengths[index] = (byte)length;
            filled[index] = true;
        }
    }
    if (filled.Any(value => !value))
        throw new Exception("Table incomplète");
    return (symbols, tableLengths);
}

var toyCounts = new long[256];
toyCounts['a'] = 8; toyCounts['b'] = 4; toyCounts['c'] = 2; toyCounts['d'] = 2;
var toyTree = BuildHuffman(toyCounts);
var toyCodes = CanonicalCodes(toyTree.Lengths);
int toyL = toyTree.Lengths.Values.Max();
var toyTable = BuildDecodeTable(toyCodes, toyTree.Lengths, toyL);
Console.WriteLine($"Code jouet : {string.Join(", ", toyCodes.Select(item => $"{(char)item.Key}={Convert.ToString((long)item.Value, 2).PadLeft(toyTree.Lengths[item.Key], '0')}"))} ; L={toyL}");
for (int index = 0; index < toyTable.Symbols.Length; index++)
    Console.WriteLine($"{Convert.ToString(index, 2).PadLeft(toyL, '0')} → {(char)toyTable.Symbols[index]}, longueur {toyTable.Lengths[index]}");

Code jouet : a=0, b=10, c=110, d=111 ; L=3


000 → a, longueur 1


001 → a, longueur 1


010 → a, longueur 1


011 → a, longueur 1


100 → b, longueur 2


101 → b, longueur 2


110 → c, longueur 3


111 → d, longueur 3


### Lecture

Un mot court remplit plusieurs indices partageant son préfixe. La longueur stockée dans chaque case est celle du mot effectivement décodé : le curseur avance de cette quantité, non de $L$.

Sur le corpus réel, la table est construite depuis les longueurs canoniques. Nous affichons son nombre d’entrées et son coût brut en mémoire ; ces deux tableaux sont reconstruits au récepteur et ne sont pas envoyés dans le flux.

In [8]:
var tableCorpus=BuildDecodeTable(codesCanoniques,longueursArbre,lMax);
long coutTable=2L*tableCorpus.Symbols.Length;
Console.WriteLine($"Corpus : L={lMax} ; table={tableCorpus.Symbols.Length:N0} entrées ; coût brut={coutTable:N0} octets ({coutTable/1024.0:F1} Kio)");
Console.WriteLine("La table est reconstruite depuis les longueurs et n’est pas transmise.");

Corpus : L=15 ; table=32,768 entrées ; coût brut=65,536 octets (64.0 Kio)


La table est reconstruite depuis les longueurs et n’est pas transmise.


### Lecture

La table réelle est reconstruite à partir des longueurs. Pour parler de taille de fichier, il faut compter les longueurs et leur association aux symboles dans l’en-tête ; la table de décodage est un coût mémoire local.

### Exercice 2 — Jusqu’où la table tient-elle ?

Écrivez `MaxLengthForBudget` pour trouver le plus grand $L$ tel que deux tableaux d’octets tiennent dans le budget. Vérifiez le résultat pour 64 Kio et dites si la table mesurée du corpus tient dans ce budget.

In [9]:
static int? MaxLengthForBudget(long budgetBytes, int bytesPerEntry = 2)
{
    // TODO étudiant : augmenter L tant que (2^L) * bytesPerEntry respecte le budget.
    return null;
}

var budgetVerdict = MaxLengthForBudget(64 * 1024);
if (budgetVerdict.HasValue)
{
    if (budgetVerdict.Value != 15)
        throw new Exception("Le budget de 64 Kio doit permettre L=15.");
    bool tableFits = coutTable <= 64 * 1024;
    if (tableFits != (lMax <= budgetVerdict.Value))
        throw new Exception("Le verdict sur la table corpus ne correspond pas au budget.");
    Console.WriteLine($"Exercice 2 : OK — L max={budgetVerdict.Value}; table du corpus {(tableFits ? "tient" : "ne tient pas")} dans 64 Kio.");
}
else
{
    Console.WriteLine("Exercice à compléter");
}

Exercice à compléter


## 5. Encoder et décoder : aller-retour exact

Le codeur écrit les mots directement dans un tampon de bits empaqueté. Le décodeur lit $L$ bits avec bourrage nul en fin de flux, indexe la table, puis avance de la longueur réelle. Le nombre de symboles attendu borne la sortie, ce qui évite d’interpréter les bits de bourrage comme des données.

In [10]:
sealed class PackedBits
{
    public List<byte> Data = new();
    public long Count;

    public void Write(BigInteger code, int length)
    {
        for (int shift = length - 1; shift >= 0; shift--)
        {
            if ((Count & 7) == 0)
                Data.Add(0);
            if (((code >> shift) & BigInteger.One) != 0)
                Data[^1] |= (byte)(1 << (7 - (int)(Count & 7)));
            Count++;
        }
    }

    public int ReadPadded(long position, int width)
    {
        int value = 0;
        for (int offset = 0; offset < width; offset++)
        {
            value <<= 1;
            long bitIndex = position + offset;
            if (bitIndex < Count)
                value |= (Data[(int)(bitIndex >> 3)] >> (7 - (int)(bitIndex & 7))) & 1;
        }
        return value;
    }
}

static PackedBits EncodeBytes(byte[] data, Dictionary<int, BigInteger> codes, Dictionary<int, int> lengths)
{
    var bits = new PackedBits();
    foreach (byte value in data)
        bits.Write(codes[value], lengths[value]);
    return bits;
}

static byte[] DecodeBytes(PackedBits bits, byte[] symbols, byte[] lengths, int maxLength, int count)
{
    var output = new byte[count];
    long position = 0;
    for (int index = 0; index < count; index++)
    {
        int tableIndex = bits.ReadPadded(position, maxLength);
        int codeLength = lengths[tableIndex];
        if (codeLength == 0 || position + codeLength > bits.Count)
            throw new Exception("Flux invalide ou table incomplète");
        output[index] = symbols[tableIndex];
        position += codeLength;
    }
    if (position != bits.Count)
        throw new Exception("Bits résiduels après le dernier symbole");
    return output;
}

var fluxEncode = EncodeBytes(extrait, codesCanoniques, longueursArbre);
var restaure = DecodeBytes(fluxEncode, tableCorpus.Symbols, tableCorpus.Lengths, lMax, extrait.Length);
var hashRestaure = Convert.ToHexString(SHA256.HashData(restaure)).ToLowerInvariant();
if (!extrait.SequenceEqual(restaure) || hashOriginal != hashRestaure)
    throw new Exception("Aller-retour ou empreinte incorrecte");
long bytesFlux = (fluxEncode.Count + 7) / 8;
Console.WriteLine($"Original : {extrait.Length:N0} octets | payload : {bytesFlux:N0} octets / {fluxEncode.Count:N0} bits");
Console.WriteLine($"Aller-retour exact : {extrait.SequenceEqual(restaure)} | SHA-256 original = restauré : {hashOriginal == hashRestaure} ({hashRestaure})");

Original : 131,072 octets | payload : 76,275 octets / 610,195 bits


Aller-retour exact : True | SHA-256 original = restauré : True (248fb4ec30c05351198a40c81969624d447fb8ad36e89d6bb5f8ce6084090704)


### Lecture

Le décodeur ne parcourt aucun arbre. La vérification compare les octets et les empreintes SHA-256, ce qui contrôle la restitution sans perte. Le tampon conserve les bits sous forme compacte : le décompte de bits n’est donc pas confondu avec une représentation textuelle coûteuse.

### Exemple 3 — Cas limites de l’alphabet

Un alphabet réduit à un seul octet reçoit par convention un code de longueur un. Un flux binaire qui contient les 256 valeurs vérifie également les indices, les longueurs et le bourrage de fin.

In [11]:
var singleCounts = new long[256];
singleCounts[0xA5] = 5;
var singleTree = BuildHuffman(singleCounts);
var singleCode = CanonicalCodes(singleTree.Lengths);
var singleTable = BuildDecodeTable(singleCode, singleTree.Lengths, 1);
byte[] singleInput = Enumerable.Repeat((byte)0xA5, 5).ToArray();
var singleBits = EncodeBytes(singleInput, singleCode, singleTree.Lengths);
var singleOutput = DecodeBytes(singleBits, singleTable.Symbols, singleTable.Lengths, 1, singleInput.Length);

var binaryInput = Enumerable.Range(0, 256).Select(value => (byte)value).ToArray();
var binaryCounts = new long[256];
foreach (byte value in binaryInput)
    binaryCounts[value]++;
var binaryTree = BuildHuffman(binaryCounts);
var binaryCode = CanonicalCodes(binaryTree.Lengths);
var binaryMaxLength = binaryTree.Lengths.Values.Max();
var binaryTable = BuildDecodeTable(binaryCode, binaryTree.Lengths, binaryMaxLength);
var binaryBits = EncodeBytes(binaryInput, binaryCode, binaryTree.Lengths);
var binaryOutput = DecodeBytes(binaryBits, binaryTable.Symbols, binaryTable.Lengths, binaryMaxLength, binaryInput.Length);

if (!singleInput.SequenceEqual(singleOutput) || !binaryInput.SequenceEqual(binaryOutput))
    throw new Exception("Échec d’un cas limite");
Console.WriteLine($"Un symbole : L={singleTree.Lengths.Values.Single()}, roundtrip={singleInput.SequenceEqual(singleOutput)}");
Console.WriteLine($"Alphabet binaire 0..255 : L={binaryMaxLength}, roundtrip={binaryInput.SequenceEqual(binaryOutput)}, bits={binaryBits.Count}");

Un symbole : L=1, roundtrip=True


Alphabet binaire 0..255 : L=8, roundtrip=True, bits=2048


### Lecture

Le cas mon symbole évite un arbre de profondeur nulle et garde un bit par symbole. Le test binaire traverse l’ensemble des indices d’octet ; il protège également contre les erreurs de signe ou de conversion lors du stockage des symboles.

### Exercice 3 — Un pas de décodage

Écrivez `NextSymbol` pour lire un index à partir de `position`, appliquer le bourrage à droite si nécessaire, puis renvoyer le symbole et la position suivante.

In [12]:
static (byte Symbol, long Next)? NextSymbol(
    PackedBits bits, long position, byte[] symbols, byte[] lengths, int maxLength)
{
    // TODO étudiant : lire maxLength bits avec ReadPadded puis avancer selon la longueur de table.
    return null;
}

var firstStep = NextSymbol(fluxEncode, 0, tableCorpus.Symbols, tableCorpus.Lengths, lMax);
if (firstStep.HasValue)
{
    byte expectedSymbol = extrait[0];
    long expectedNext = longueursArbre[expectedSymbol];
    if (firstStep.Value.Symbol != expectedSymbol || firstStep.Value.Next != expectedNext)
        throw new Exception("Le premier pas doit restituer le premier octet et sa longueur de code.");
    Console.WriteLine($"Exercice 3 : OK — octet={firstStep.Value.Symbol}, position suivante={firstStep.Value.Next}");
}
else
{
    Console.WriteLine("Exercice à compléter");
}

Exercice à compléter


## 6. Débits : mesures locales et références publiées

Nous comparons les primitives C# pédagogiques par médiane de plusieurs répétitions. Le flux reste empaqueté en mémoire ; les mesures dépendent du runtime et de la machine. Les chiffres publiés de Żukowski, *PivCo-Huffman* v1.0 (2026), section 1.2, proviennent d’implémentations C/assembleur, d’autres corpus et d’autres machines : ils servent de repère de production, pas de comparaison contrôlée.

In [13]:
int repetitions = 5;
var encTimes = new List<double>();
var decTimes = new List<double>();
for (int repetition = 0; repetition < repetitions; repetition++)
{
    var stopwatch = Stopwatch.StartNew();
    var encoded = EncodeBytes(extrait, codesCanoniques, longueursArbre);
    stopwatch.Stop();
    encTimes.Add(stopwatch.Elapsed.TotalSeconds);

    stopwatch.Restart();
    var decoded = DecodeBytes(encoded, tableCorpus.Symbols, tableCorpus.Lengths, lMax, extrait.Length);
    stopwatch.Stop();
    decTimes.Add(stopwatch.Elapsed.TotalSeconds);
    if (!decoded.SequenceEqual(extrait))
        throw new Exception("Contrôle benchmark échoué");
}

static double Median(List<double> values)
{
    var ordered = values.OrderBy(value => value).ToArray();
    return ordered[ordered.Length / 2];
}

double sizeMiB = (double)extrait.Length / (1024 * 1024);
double encRate = sizeMiB / Median(encTimes);
double decRate = sizeMiB / Median(decTimes);
Console.WriteLine($"Méthodologie : {extrait.Length:N0} octets, {repetitions} répétitions, médiane, C# BCL, machine locale");
Console.WriteLine($"Encodage : {encRate:F2} Mio/s | Décodage table : {decRate:F2} Mio/s");
long bytesFlux = (fluxEncode.Count + 7) / 8;
long denseHeaderBytes = 256;
long sparseHeaderBytes = 2L * longueursArbre.Count;
Console.WriteLine($"Coût brut : payload {bytesFlux:N0} octets + en-tête dense (256 longueurs sur 1 octet) {denseHeaderBytes} octets = {bytesFlux + denseHeaderBytes:N0} octets, hors framing.");
Console.WriteLine($"En-tête creux explicite (octet symbole + octet longueur pour {longueursArbre.Count} symboles) : {sparseHeaderBytes} octets, hors framing ; la table locale coûte {coutTable:N0} octets et ne voyage pas.");
Console.WriteLine("Références citées par Żukowski (2026, §1.2) — débits publiés, autres corpus et machines :");
Console.WriteLine("Huff0 C, proba80 / M4 : enc 1334 Mo/s ; déc 2929 Mo/s");
Console.WriteLine("Oodle C+assembleur, proba80 / M4 : enc 1740 Mo/s ; déc 3343 Mo/s");
Console.WriteLine("Huff0 C, prose_pride / c8i : enc 1267 Mo/s ; déc 1758 Mo/s");
Console.WriteLine("Oodle C+assembleur, prose_pride / c8i : enc 1364 Mo/s ; déc 2120 Mo/s");

Méthodologie : 131,072 octets, 5 répétitions, médiane, C# BCL, machine locale


Encodage : 1.47 Mio/s | Décodage table : 6.41 Mio/s


Coût brut : payload 76,275 octets + en-tête dense (256 longueurs sur 1 octet) 256 octets = 76,531 octets, hors framing.


En-tête creux explicite (octet symbole + octet longueur pour 88 symboles) : 176 octets, hors framing ; la table locale coûte 65,536 octets et ne voyage pas.


Références citées par Żukowski (2026, §1.2) — débits publiés, autres corpus et machines :


Huff0 C, proba80 / M4 : enc 1334 Mo/s ; déc 2929 Mo/s


Oodle C+assembleur, proba80 / M4 : enc 1740 Mo/s ; déc 3343 Mo/s


Huff0 C, prose_pride / c8i : enc 1267 Mo/s ; déc 1758 Mo/s


Oodle C+assembleur, prose_pride / c8i : enc 1364 Mo/s ; déc 2120 Mo/s


### Lecture

La sortie réunit nos mesures locales et les débits publiés de Żukowski, section 1.2. Ces derniers concernent Huff0 en C et Oodle en C/assembleur sur `proba80` (M4) et `prose_pride` (c8i) ; corpus et machines diffèrent, ils sont donc un repère qualitatif de production, pas un benchmark comparable. Dans notre implémentation, le coût des opérations de haut niveau et de l’écriture bit à bit reste visible.

Le tableau de longueurs et la table de décodage n’ont pas le même rôle : les longueurs font partie des informations nécessaires au récepteur, tandis que la table est reconstruite localement. Un format réel doit aussi coder l’alphabet, les longueurs, le nombre de bits utiles et la taille du bloc. Le poids de cet en-tête peut être important pour de petits messages.

Huffman est optimal pour un code préfixe symbole par symbole donné l’histogramme. Il ne modélise ni les mots répétés ni le contexte ; LZ77 et les modèles adaptatifs exploitent d’autres régularités. Les débits affichés dépendent des primitives de cette implémentation et de l’environnement, et ne doivent pas être épinglés dans cette prose.

## À retenir

| Notion | Ce qu’elle dit | Où la voir |
|---|---|---|
| Forme canonique | Les longueurs suffisent à reconstruire les mots | Section 3 |
| Kraft complet | La table $2^L$ est remplie sans trou | Sections 3–4 |
| Table directe | Un accès indexé échange mémoire contre temps | Section 4 |
| Aller-retour | Les octets et l’empreinte sont identiques après décodage | Section 5 |
| Coût complet | En-tête, bourrage, données et table locale sont distingués | Sections 5–6 |

Ce carnet a construit un code Huffman depuis un histogramme d’octets, reconstruit les mots canoniques depuis leurs longueurs et décodé par table. Le flux compressé est empaqueté et le contrôle porte sur le corpus complet. Les limites viennent du modèle symbole par symbole, du coût d’en-tête et de la croissance exponentielle de la table.

Références : E. S. Schwartz et B. Kallick, « Generating a canonical prefix encoding », *Communications of the ACM* 7(3), 1964 · M. Żukowski, *PivCo-Huffman* v1.0, 2026, section 1.2.

## Vérifiez votre compréhension

1. Pourquoi les longueurs suffisent-elles à reconstruire les mots canoniques ?

<details><summary>Réponse</summary>

Le tri canonique et l’incrément/décalage déterminent un seul mot pour chaque symbole.

</details>

2. Quel compromis la table $2^L$ réalise-t-elle ?

<details><summary>Réponse</summary>

Elle permet de décoder avec un accès indexé, au coût de deux tableaux qui croissent exponentiellement avec $L$.

</details>

3. Quelles composantes doivent entrer dans l’évaluation d’un fichier compressé ?

<details><summary>Réponse</summary>

Le payload en bits/bytes, le bourrage et l’en-tête décrivant alphabet, longueurs et longueur utile. La table peut être reconstruite et reste un coût mémoire local.

</details>

## Pour aller plus loin

- [Compression-03 — LZ77](Compression-03-LZ77-Python.ipynb) représente les répétitions par des références.
- [Compression-01b — Kraft–McMillan en Lean](Compression_lean/Compression-01b-Kraft-McMillan-Lean.ipynb) formalise les bornes des codes préfixes.
- [README de la série Compression](README.md) présente le parcours complet.